# Diagnóstico da Base de Modelagem

Este notebook avalia a base disponível para o problema de classificação da Fase 3.

O objetivo é entender a unidade de modelagem, o target, a cobertura temporal,
a distribuição das classes e as variáveis atualmente disponíveis antes da
construção da base analítica de Machine Learning.

A unidade de modelagem considerada é um aluno em determinado ano de avaliação.

O target é a variável `alfabetizado`, onde:

- `0`: aluno não alfabetizado;
- `1`: aluno alfabetizado.

## 1. Configuração

Nesta etapa são importadas as bibliotecas utilizadas no diagnóstico e definida a tabela principal de alunos produzida na Fase 2.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
tabela_alunos = "workspace.silver.alunos"

df_alunos = spark.table(tabela_alunos)

print(f"Total de registros: {df_alunos.count():,}")

df_alunos.printSchema()

## 2. Visão Geral da Base

Primeiro são avaliados o volume de registros, a cobertura temporal e a granularidade dos dados disponíveis.

In [0]:
display(df_alunos.limit(20))

In [0]:
display(
    df_alunos.groupBy("ano")
    .agg(
        F.count("*").alias("qtd_registros"),
        F.countDistinct("id_aluno").alias("qtd_alunos"),
    )
    .orderBy("ano")
)

## 3. Distribuição do Target

O target do modelo é a variável `alfabetizado`.

Nesta etapa é avaliada a proporção entre alunos alfabetizados e não alfabetizados em cada ano.

In [0]:
df_balanceamento = (
    df_alunos.filter(F.col("alfabetizado").isNotNull())
    .groupBy("ano", "alfabetizado")
    .count()
)


window_ano = Window.partitionBy("ano")

df_balanceamento = df_balanceamento.withColumn(
    "total_ano", F.sum("count").over(window_ano)
).withColumn("percentual", F.round(F.col("count") / F.col("total_ano") * 100, 2))

display(df_balanceamento.orderBy("ano", "alfabetizado"))

In [0]:
display(
    df_alunos.groupBy("ano")
    .agg(
        F.count("*").alias("total_registros"),
        F.sum(F.when(F.col("alfabetizado").isNull(), 1).otherwise(0)).alias(
            "target_nulo"
        ),
    )
    .withColumn(
        "pct_target_nulo",
        F.round(F.col("target_nulo") / F.col("total_registros") * 100, 2),
    )
    .orderBy("ano")
)

## 4. Validação do Target

Embora a variável `alfabetizado` não possua valores nulos, é necessário verificar
se o rótulo representa uma avaliação efetivamente realizada.

Registros de alunos ausentes ou com caderno não preenchido podem possuir
`alfabetizado = 0`, mesmo sem resultado observado na avaliação.

Esses casos devem ser analisados separadamente para evitar que ausência na prova
seja interpretada pelo modelo como não alfabetização.

In [0]:
display(
    df_alunos.groupBy(
        "ano",
        "presenca",
        "presenca_descricao",
        "preenchimento_caderno",
        "preenchimento_caderno_descricao",
        "alfabetizado",
        "alfabetizado_descricao",
    )
    .count()
    .orderBy("ano", "presenca", "preenchimento_caderno", "alfabetizado")
)

In [0]:
df_avaliados = df_alunos.filter(
    (F.col("presenca") == "1") & (F.col("preenchimento_caderno") == "1")
)

print(f"Base original: {df_alunos.count():,}")
print(f"Alunos avaliados: {df_avaliados.count():,}")

In [0]:
total = df_alunos.count()
avaliados = df_avaliados.count()

print(f"Percentual elegível: {avaliados / total * 100:.2f}%")

Público Elegível - Fez a Prova:

In [0]:
df_balanceamento_avaliados = df_avaliados.groupBy(
    "ano", "alfabetizado", "alfabetizado_descricao"
).count()

window_ano = Window.partitionBy("ano")

df_balanceamento_avaliados = df_balanceamento_avaliados.withColumn(
    "total_ano", F.sum("count").over(window_ano)
).withColumn("percentual", F.round(F.col("count") / F.col("total_ano") * 100, 2))

display(df_balanceamento_avaliados.orderBy("ano", "alfabetizado"))

## 5. Análise das Variáveis Categóricas Atuais

São avaliadas as categorias disponíveis nas principais variáveis já existentes, buscando identificar quais campos possuem variação suficiente para contribuir com o modelo.

In [0]:
display(
    df_avaliados.groupBy("serie", "serie_descricao").count().orderBy(F.desc("count"))
)

In [0]:
display(df_avaliados.groupBy("rede", "rede_descricao").count().orderBy(F.desc("count")))

## 6. Análise de Repetição de Alunos

In [0]:
alunos_2023 = (
    df_avaliados
    .filter(F.col("ano") == 2023)
    .select("id_aluno")
    .distinct()
)

alunos_2024 = (
    df_avaliados
    .filter(F.col("ano") == 2024)
    .select("id_aluno")
    .distinct()
)

alunos_em_ambos = (
    alunos_2023
    .join(
        alunos_2024,
        on="id_aluno",
        how="inner"
    )
    .count()
)

print(
    f"Alunos presentes em 2023 e 2024: {alunos_em_ambos:,}"
)

## 7. Target por Rede

In [0]:
display(
    df_avaliados.groupBy(
        "ano", "rede", "rede_descricao", "alfabetizado", "alfabetizado_descricao"
    )
    .count()
    .orderBy("ano", "rede", "alfabetizado")
)

## 8. Identificação dos Alunos entre os Anos

Foi encontrada uma quantidade elevada de `id_aluno` presente simultaneamente em 2023 e 2024.

Como todos os registros correspondem ao 2º ano do Ensino Fundamental, é necessário verificar se esse identificador representa longitudinalmente o mesmo aluno ou se pode ser reutilizado entre diferentes edições da avaliação.

In [0]:
df_2023 = df_avaliados.filter(F.col("ano") == 2023).select(
    "id_aluno",
    F.col("id_municipio").alias("municipio_2023"),
    F.col("id_escola").alias("escola_2023"),
    F.col("rede").alias("rede_2023"),
)

df_2024 = df_avaliados.filter(F.col("ano") == 2024).select(
    "id_aluno",
    F.col("id_municipio").alias("municipio_2024"),
    F.col("id_escola").alias("escola_2024"),
    F.col("rede").alias("rede_2024"),
)

df_ids_repetidos = df_2023.join(df_2024, on="id_aluno", how="inner")


resumo_ids = df_ids_repetidos.agg(
    F.count("*").alias("ids_em_ambos_anos"),
    F.sum(
        F.when(F.col("municipio_2023") == F.col("municipio_2024"), 1).otherwise(0)
    ).alias("mesmo_municipio"),
    F.sum(F.when(F.col("escola_2023") == F.col("escola_2024"), 1).otherwise(0)).alias(
        "mesma_escola"
    ),
    F.sum(F.when(F.col("rede_2023") == F.col("rede_2024"), 1).otherwise(0)).alias(
        "mesma_rede"
    ),
)

display(resumo_ids)

In [0]:
display(df_ids_repetidos.limit(30))

In [0]:
df_alunos.filter(F.col("id_aluno") == "22026659").display()

## 9. Inventário Inicial de Features
As variáveis disponíveis são classificadas conforme seu potencial de uso no modelo, considerando variabilidade, cardinalidade e risco de data leakage.

In [0]:
colunas_candidatas = [
    "id_municipio",
    "id_escola",
    "rede",
    "caderno",
    "serie",
    "peso_aluno",
]

for coluna in colunas_candidatas:
    resumo = df_avaliados.agg(
        F.countDistinct(coluna).alias("valores_distintos"),
        F.sum(F.when(F.col(coluna).isNull(), 1).otherwise(0)).alias("nulos"),
    ).collect()[0]

    print(
        f"{coluna}: "
        f"{resumo['valores_distintos']} valores distintos | "
        f"{resumo['nulos']} nulos"
    )

## 10. Conclusões do Diagnóstico

O diagnóstico inicial permitiu definir a população e as principais restrições da base de modelagem.

A população elegível é formada pelos alunos presentes e com prova preenchida, totalizando 3.354.661 observações, aproximadamente 86,7% da base original.

O target apresenta distribuição relativamente equilibrada entre as classes, com aproximadamente 40% de alunos não alfabetizados e 60% alfabetizados.

* A variável `serie` possui apenas um valor e, portanto, não será utilizada como feature. <br>
* A rede de ensino apresenta variação suficiente e será mantida comovariável candidata. <br>
* Os campos `presenca`, `preenchimento_caderno` e `proficiencia` não serão utilizados como features, pois representam informações conhecidas durante ou após a avaliação e poderiam introduzir data leakage. <br>
* Os identificadores de município e escola possuem alta cardinalidade e serão utilizados principalmente como chaves para enriquecimento com informações territoriais, socioeconômicas e educacionais. <br>
* Também foi identificado que `id_aluno` não deve ser interpretado como identificador único entre 2023 e 2024. Apesar de muitos códigos aparecerem nos dois períodos, a correspondência territorial e escolar é baixa. <br>

Dessa forma, a estratégia inicial será utilizar os dados de 2023 para desenvolvimento, treinamento e validação dos modelos, mantendo 2024 como conjunto de teste temporal final.

O próximo passo será enriquecer a base com variáveis externas que representem o contexto socioeconômico dos municípios e as características educacionais das escolas.